In [2]:
import requests
import json
from bs4 import BeautifulSoup
from urllib.parse import unquote
import os
from dotenv import load_dotenv
import pandas as pd

In [ ]:
# 자전거 사고 다발 지역
BASE_URL = 'http://apis.data.go.kr/B552061/frequentzoneBicycle/getRestFrequentzoneBicycle'
KEY = 'TB2d6bj%2F%2BY1ChsQButXeXxrwO3PhNqPSiGkbjQuYmaYgpAPY7d5S39di4nEP%2FmLvl9jvGVBYexaMd1O%2BxxqOHQ%3D%3D'
# unquote('TB2d6bj%2F%2BY1ChsQButXeXxrwO3PhNqPSiGkbjQuYmaYgpAPY7d5S39di4nEP%2FmLvl9jvGVBYexaMd1O%2BxxqOHQ%3D%3D')

In [ ]:
# 1번 데이터, 자전거 사고 다발지역 api 크롤링 및 sql 적재

import pandas as pd
import requests
from dotenv import load_dotenv
import os
import time

BASE_URL = 'http://apis.data.go.kr/B552061/frequentzoneBicycle/getRestFrequentzoneBicycle'
load_dotenv('11.env')


code_df = pd.read_csv(
    '법정동코드전체자료.txt',
    sep='\t',
    encoding='euc-kr',
    dtype=str
)

sgg_df = code_df[
    (code_df['법정동코드'].str[2:5] != '000') &
    (code_df['법정동코드'].str[5:] == '00000') &
    (code_df['폐지여부'] == '존재')
].copy()

sgg_df['siDo'] = sgg_df['법정동코드'].str[0:2].astype(int)
sgg_df['guGun'] = sgg_df['법정동코드'].str[2:5].astype(int)
sgg_df['sido_sgg_nm'] = sgg_df['법정동명']

region_list = sgg_df[['siDo', 'guGun', 'sido_sgg_nm']].to_dict('records')

years = ['2022', '2023', '2024', '2025']

result = []
for year in years:
    for region in region_list:
        params = {
            'serviceKey': 'TB2d6bj/+Y1ChsQButXeXxrwO3PhNqPSiGkbjQuYmaYgpAPY7d5S39di4nEP/mLvl9jvGVBYexaMd1O+xxqOHQ==',
            'searchYearCd': year,
            'siDo': region['siDo'],
            'guGun': region['guGun'],
            'type': 'json',
            'numOfRows': 100,
            'pageNo': 1
        }

        response = requests.get(BASE_URL, params=params)
        try:
            data = response.json()
        except ValueError:
            print(f"[경고] {year} / {region['sido_sgg_nm']} JSON 파싱 실패")
            continue

        items = data.get("items", {}).get("item", [])
        if not items:
            continue
        if isinstance(items, dict):
            items = [items]

        for item in items:
            result.append({
                '연도': year, 
                "법정동코드": item["bjd_cd"],
                "시도시군구명": item["sido_sgg_nm"],
                "지점명": item["spot_nm"],
                '사고건수': item["occrrnc_cnt"],
                "사상자수": item["caslt_cnt"],
                "사망자수": item["dth_dnv_cnt"],
                "중상자수": item["se_dnv_cnt"],
                "경상자수": item["sl_dnv_cnt"],
                "부상신고자수": item["wnd_dnv_cnt"],
                '경도': float(item['lo_crd']),
                '위도': float(item['la_crd'])
            })

    
        time.sleep(0.1)

print(len(result))

926


In [ ]:
import os
print(os.getcwd())      
print(os.listdir())

c:\Users\User\Desktop\campus\workspace
['.git', '.venv', '.venv-1', '.vscode', '0819 필기.ipynb', '0820과제.ipynb', '0831연습.ipynb', '11.env', 'asos_108_20240101_20240110.csv', 'benchmark.py', 'bench_server.py', 'bikeraw.sql', 'cars.csv', 'env (1)', 'miniproject2', 'mini_project2_guide.md', 'musinsa.csv', 'my_package', 'output', 'project', 'quotes_project', 'raw.sql', 'samsung_1y.csv', 'TB_PTP_SHBK_RNTL_SPOT_INFO.csv', 'urls.txt', 'vibe_top100.csv', 'weather.env', '__pycache__', '과제1.png', '과제1_답.png', '과제2.png', '더미파일', '데이터관리.ipynb', '마크다운 문법.md', '법정동코드전체자료.txt', '연습장들']


In [ ]:
os.chdir(r'C:\Users\User\Desktop\campus\workspace')

print(os.getcwd())

In [ ]:
df = pd.DataFrame(result)

print(df.shape)
print(df.head())
df

In [ ]:
# mysql 생성하기

import os
import pymysql
from pymysql.cursors import DictCursor
from dotenv import load_dotenv

load_dotenv('11.env')
from sqlalchemy import create_engine

host = os.getenv('DB_HOST')
port = 3306
password = os.getenv('DB_PASSWORD')

engine = create_engine(f'mysql+pymysql://root:{password}@{host}:{port}/bike_accident_db')

df = pd.read_sql('SELECT * FROM raw_item', engine)
DB_CONFIG = {
    'host': os.getenv('DB_HOST'),
    'port': int(os.getenv('DB_PORT')),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'database': 'bike_accident_db',
    'charset': 'utf8mb4',
    'cursorclass': DictCursor
}

def connect():
    conn = pymysql.connect(**DB_CONFIG)
    return conn

conn = connect()
conn.close()

In [ ]:
# mysql에 적재하기
df.to_sql(
    "raw_item",
    con=engine,
    if_exists="replace",
    index=False
)

926

In [ ]:
#mart 만들기

# 사고건수 랭킹

CREATE TABLE accident_rank AS
SELECT 
    연도,
    지점명,
    사고건수,
    부상신고자수,
    위도,
    경도
FROM raw_item
ORDER BY 사고건수 DESC
LIMIT 10;

# 부상위험율 랭킹

CREATE TABLE danger_rank AS
SELECT 
    연도,
    지점명,
    사고건수,
    사상자수,
    중상자수,
    위도,
    경도
FROM raw_item
ORDER BY 사상자수 DESC
LIMIT 10;

In [ ]:
pip install pandas sqlalchemy pymysql python-dotenv

In [ ]:
# 2번 데이터, 대여소 목록 csv 적재하기
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv('11.env')

DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT', '3306')
DB_NAME = os.getenv('DB_NAME')


df = pd.read_csv(
    'TB_PTP_SHBK_RNTL_SPOT_INFO.csv',
    encoding='utf-8-sig'
)

print(df.shape)   
print(df.head()) 

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4"
)

df.to_sql(
    name='bike_rental_spot_info', 
    con=engine,
    if_exists='replace',  
    index=False           
)


In [ ]:
# 3번 데이터, 중증외상자 자전거 헬멧 착용율

df = pd.read_csv(
    '중증외상__자전거사고_헬멧_착용률.csv',
    encoding='utf-8-sig',
    skiprows=2,         
    header=None        
)

df.columns = [
    '시도',
    '구분1',        
    '구분2',      
    '착용_건수',
    '착용_분율',
    '미착용_건수',
    '미착용_분율',
    '미상_건수',
    '미상_분율'
]

df = df.replace('-', pd.NA)

# 5. 숫자 컬럼들을 실제 숫자(float)로 변환
숫자컬럼 = ['착용_건수', '착용_분율', '미착용_건수', '미착용_분율', '미상_건수', '미상_분율']
for col in 숫자컬럼:
    df[col] = pd.to_numeric(df[col], errors='coerce')  

print(df.shape)
print(df.head(10))

df.to_sql(
    name='helmet_rate',  
    con=engine,
    if_exists='replace',
    index=False
)


In [ ]:
# 사고 다발생 구역이 자전거 대여소 근방인지 확인하기


accident_df = pd.read_sql("SELECT * FROM accident_rank", con=engine)
rental_df = pd.read_csv('TB_PTP_SHBK_RNTL_SPOT_INFO.csv', encoding='utf-8-sig')

LAT_DIFF = 0.009
LON_DIFF = 0.011

결과_존재여부 = []
결과_대여소명 = []


for i in range(len(accident_df)):
    사고_위도 = accident_df.loc[i, '위도']
    사고_경도 = accident_df.loc[i, '경도']

    찾은_대여소 = None 

    for j in range(len(rental_df)):
        대여소_위도 = rental_df.loc[j, 'CMWNR_BIKE_RNTL_SMALL_LTTD']
        대여소_경도 = rental_df.loc[j, 'CMWNR_BIKE_RNTL_SMALL_LNGTD']
        대여소_이름 = rental_df.loc[j, 'CMWNR_BIKE_RNTL_SMALL_NM']

        위도차이 = abs(사고_위도 - 대여소_위도)
        경도차이 = abs(사고_경도 - 대여소_경도)

        if 위도차이 <= LAT_DIFF and 경도차이 <= LON_DIFF:
            찾은_대여소 = 대여소_이름
            break  # 하나 찾았으면 더 볼 필요 없이 멈춤


    if 찾은_대여소 is not None:
        결과_존재여부.append('있음')
        결과_대여소명.append(찾은_대여소)
    else:
        결과_존재여부.append('없음')
        결과_대여소명.append('-')


accident_df['대여소_존재여부'] = 결과_존재여부
accident_df['가까운_대여소명'] = 결과_대여소명


print(accident_df[['지점명', '위도', '경도', '대여소_존재여부', '가까운_대여소명']])
